# CV 架构与融合算子

本节先认识 Atlas A2/A3 上 Cube Core 与 Vector Core 分离部署的架构，再由矩阵结果需要交给矢量计算这一数据依赖引出 CV 融合。

本节学习大纲：

- 理解 A2/A3 的 AIC、AIV 与分离模式。
- 理解 Cube 和 Vector 的典型数据流。
- 理解 CV 融合的定义、执行方式与价值。
- 掌握 CV 融合的适用条件和验证边界。

---

## 1. A2/A3 的 AIC/AIV 分离架构

### 1.1 Cube Core 与 Vector Core

Atlas A2/A3 的 AI Core 工作在分离模式：矩阵计算单元和矢量计算单元分别部署在 Cube Core 与 Vector Core 上，并分别具有独立的 Scalar 调度单元。分离模式下，一组 Cube Core 和 Vector Core 按 `1:N` 组合成一个 AI Core，具体的 `N` 由硬件平台信息决定。

在这一组合中：

- **AIC** 是 Cube Core，负责矩阵计算。
- **AIV** 是 Vector Core，负责矢量计算。
- 两侧的 Scalar 分别负责本核的指令发射和流程控制。

<img src="./images/a2-separated-architecture.png" alt="Atlas A2/A3 AIC 与 AIV 分离模式" width="700px">

*图 5-1 Atlas A2/A3 的 AIC/AIV 分离模式*

### 1.2 Cube 与 Vector 的典型数据流

Cube 和 Vector 使用不同的片上存储完成计算：

```text
Cube：  GM -> L1 -> L0A/L0B -> Cube -> L0C -> FixPipe -> GM
Vector：GM -> UB -> Vector -> UB -> GM
```

其中，L0A/L0B 保存 Cube 的输入，L0C 保存 Cube 的结果和中间结果；Vector 的源数据与目的数据位于 Unified Buffer（UB）。因此，当 Matmul 后接矢量计算时，需要按平台支持的数据路径将 Matmul 计算结果搬运到 Vector 核。这个跨计算单元的数据依赖正是 CV 融合需要组织的核心问题。

## 2. 从分离架构到 CV 融合

### 2.1 CV 融合的定义

CV 融合把存在数据依赖的 Cube 计算和 Vector 计算放入一个算子 Kernel 中。融合后的功能必须与原来多个独立算子的组合等价。典型结构如下：

```text
D = Matmul(A, B) + Bias     # AIC 上的 Cube 计算
C = Activation(D)           # AIV 上的 Vector 计算
```

CV 融合并没有改变 AIC 与 AIV 的职责，而是在一个 Kernel 内完成 Matmul 计算、将 Matmul 结果搬运到 Vector 核以及 Vector 矢量计算。使用 Matmul 高阶 API 时，开发者按 Matmul 分片获取结果并执行 Vector 后处理；AIC/AIV 代码隔离和核间同步由框架完成。

### 2.2 分片流水

独立算子需要先完成整个矩阵算子，再调度矢量算子。CV 融合可以把输出切分为多个分片：AIC 生成一个分片后，AIV 即可处理该分片；流水稳定后，AIC 和 AIV 可以处理不同分片。

<img src="./images/cv-fusion-dataflow.png" alt="AIC 到 AIV 的 CV 融合数据流" width="700px">

*图 5-2 AIC 到 AIV 的 CV 融合数据流*

这种组织方式带来两类明确变化：

- 多个独立算子合并为一个 Kernel，减少算子间的调度。
- 完整中间矩阵不再作为两个独立 Kernel 的交接边界，可以按分片组织 Cube 与 Vector 流水。

流水能否产生性能收益取决于 Shape、分片、片上空间和两阶段耗时，不能仅凭完成融合就推导出固定加速比。

## 3. CV 融合的适用条件

### 3.1 存在直接的数据依赖

LeakyReLU 等 Vector 计算需要直接使用前一个 Matmul 计算的结果。两种计算之间存在这种直接数据依赖时，才能围绕 Matmul 中间结果组织 CV 分片流水。

### 3.2 中间结果能够分片处理

Cube 输出分片必须能够作为 Vector 阶段的计算单位。以 Matmul 高阶 API 为例，每次迭代产生一个 `base_m * base_n` 的输出块，Vector 激活和写回也围绕这个块进行。

### 3.3 片上缓冲区能够容纳当前分片

Vector 计算需要 LocalTensor 保存当前结果分片。设计 Tiling 时，需要为该分片预留足够的 UB 空间；分片过大而无法放入可用空间时，需要调整 `base_m`、`base_n` 或缓冲规划。

### 3.4 融合前后功能等价

融合必须保留完整数学语义、输出布局和数据类型要求。正确性验证应使用不复用融合 Kernel 调度逻辑的独立表达式，并在当前 Shape、数据类型和 Tiling 下比较输出。

### 3.5 在目标规格上测量收益

CV 融合提供分片流水和减少调度的条件，但实际收益仍需在目标硬件、目标 Shape 与 Tiling 上测量。若分片不合理、Vector 阶段过长或片上资源不足，流水效果会受到影响。

---

## 课后练习

### 第 1 题（多选）

关于 Atlas A2/A3 分离模式，哪些说法正确？

A. AIC 是负责矩阵计算的 Cube Core。  
B. AIV 是负责矢量计算的 Vector Core。  
C. Cube Core 与 Vector Core 分别具有独立的 Scalar 调度单元。  
D. Vector 的源数据和目的数据直接位于 L0C。

### 第 2 题（多选）

哪些条件适合组织 CV 融合？

A. Vector 计算直接使用 Cube 计算结果。  
B. Matmul 中间结果能够按块搬运到 Vector 核。  
C. 当前分片能够放入可用的片上缓冲区。  
D. 不需要检查融合前后的数学语义。

### 第 3 题（判断）

只要把 Cube 和 Vector 计算写入一个 Kernel，就能保证在所有 Shape 和 Tiling 下获得相同的性能收益。

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/05.02_answer.txt